In [2]:
# Step 1: Connect LangChain to Ollama
# Goal: Create a LangChain llm object that talks to local Ollama
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:0.6b",
    temperature=0
)

print(type(llm))
print("Langchain is connected to Ollama")

<class 'langchain_ollama.chat_models.ChatOllama'>
Langchain is connected to Ollama


In [ ]:
# Step 2: Send a message using LangChain
# Goal: Compare with ollama.chat() -- same result, different syntax
from langchain_core.messages import HumanMessage
response = llm.invoke([
    HumanMessage(content="What is the capital of India? Answer in one sentence. /no_think")
])
print(response)
print()
print(type(response))
print("Content:", response.content)

content='The capital of India is New Delhi.' additional_kwargs={} response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-10-15T15:58:14.6441691Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13203780600, 'load_duration': 8237448000, 'prompt_eval_count': 26, 'prompt_eval_duration': 385311000, 'eval_count': 106, 'eval_duration': 4492044000, 'logprobs': None, 'model_name': 'qwen3:0.6b', 'model_provider': 'ollama'} id='lc_run--01a14048-e4d3-7f21-a76c-f95eec0c899d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 26, 'output_tokens': 106, 'total_tokens': 132}

<class 'langchain_core.messages.ai.AIMessage'>
Content: The capital of India is New Delhi.


<h3 style="text-align: center; color: red;">Prompt Templates</h3>

In [ ]:
# Step 3: Create a PromptTemplate
# Goal: Build a reusable prompt with a placeholder
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep answers to 2 sentences."),
    ("human", "Tell me about {city}. What is it famous for? /no_think")
])
# see What the templates look like
print(prompt)

input_variables=['city'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant. Keep answers to 2 sentences.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['city'], input_types={}, partial_variables={}, template='Tell me about {city}. What is it famous for? /no_think'), additional_kwargs={})]


In [5]:
# Step 4a: Use the template -- fill in the placeholder
# Goal: See how the placeholder gets replaced
formatted = prompt.invoke({"city": "Mysore"})
print(formatted)

messages=[SystemMessage(content='You are a helpful assistant. Keep answers to 2 sentences.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about Mysore. What is it famous for? /no_think', additional_kwargs={}, response_metadata={})]


In [6]:
# Step 4b: Send the formatted prompt to the LLM
# Goal: Get a response using our template
response = llm.invoke(formatted)
print("Mysore:", response.content)

Mysore: Mysore is a city in India famous for its historical Mysore Fort, built in the 16th century, and its tea industry, which is a major economic driver.


In [7]:
# Step 4c: Same template, different city
# Goal: Reuse -- write once, use many times
formatted_2 = prompt.invoke({"city": "Jaipur"})
response_2 = llm.invoke(formatted_2)
print("Jaipur:", response_2.content)


Jaipur: Jaipur is a city in India famous for its beautiful architecture, especially the Jaipur Fort, and the Jaipur River, which is a major river in the region.


<h3 style="text-align: center; color: #fa0909;"> LCEL -- The Pipe Syntax</h3>

In [9]:
# Step 5: LCEL chain -- connect prompt + LLM with a pipe
# Goal: See the pipe operator | in action
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="qwen3:0.6b",
    temperature=0
)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer in exactly 2 sentences."),
    ("human", "What is {topic}? /no_think")
])
# LCEL chain -- connecting prompt and llm with a pipe
chain = prompt | llm
# use the chain
result = chain.invoke({"topic": "machine learning"})

print(result.content)

Machine learning is a type of artificial intelligence where algorithms learn patterns from data to make decisions or predictions. It involves training models on datasets to improve accuracy over time.


In [10]:
# Step 6: Add an output parser -- get clean text
# Goal: StrOutputParser extracts just the text
from langchain_core.output_parsers import StrOutputParser
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"topic": "deep learning"})
print(result)
print("Type:", type(result))

Deep learning is a subset of machine learning that involves training neural networks to recognize patterns in data, enabling tasks like image recognition or natural language processing.
Type: <class 'langchain_core.messages.base.TextAccessor'>


In [11]:
# Step 7a: Practical chain -- set up the product extractor
# Goal: Build a chain that extracts product info as JSON
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract product info from the text. Return ONLY a JSON object with keys: name, price, category. No other text."),
    ("human", "{text} /no_think")
])
chain = prompt | llm | StrOutputParser()
print("Chain ready")

# Step 7b: Test the product extractor chain
# Goal: Run the same chain on different products
print(chain.invoke({"text": "The Samsung Galaxy S24 costs Rs. 74,999 and is a smartphone."}))
print(chain.invoke({"text": "The Sony WH-1000XM5 headphones are available for Rs. 29,990."}))


Chain ready
{
  "name": "Samsung Galaxy S24",
  "price": "74999",
  "category": "smartphone"
}
{
  "name": "Sony WH-1000XM5",
  "price": "29990",
  "category": "headphones"
}


<h3 style="text-align: center; color: #097200;">Output Parsers</h3>

In [12]:
# Step 8: JsonOutputParser -- get a Python dictionary
# Goal: Automatic JSON parsing (no manual json.loads)
from langchain_core.output_parsers import JsonOutputParser
prompt = ChatPromptTemplate.from_messages([
    ("system", "Return ONLY valid JSON with keys: name, population_millions (number), state. No other text."),
    ("human", "Extract city info: {city_text} /no_think")
])
json_parser = JsonOutputParser()
chain = prompt | llm | json_parser
result = chain.invoke({
    "city_text": "Bangalore is in Karnataka with about 12 million people."
})
print(result)
print("Type:", type(result))
print("Name:",result["name"])

{'name': 'Bangalore', 'population_millions': 12, 'state': 'Karnataka'}
Type: <class 'dict'>
Name: Bangalore


In [13]:
# Step 9a: Define a Pydantic model for validation
# Goal: Describe the EXACT shape we expect from the LLM
from pydantic import BaseModel, Field
class CityInfo(BaseModel):
    """Information about an Indian city."""
    name: str = Field(description="Name of the city")
    state: str = Field(description="State the city is in")
    population_millions: float = Field(description="Population in millions")
    famous_for: str = Field(description="What the city is famous for")

In [14]:
# Step 9b: Create a PydanticOutputParser
# Goal: See what format instructions it generates
from langchain_core.output_parsers import PydanticOutputParser
pydantic_parser = PydanticOutputParser(pydantic_object=CityInfo)
print(pydantic_parser.get_format_instructions())


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Information about an Indian city.", "properties": {"name": {"description": "Name of the city", "title": "Name", "type": "string"}, "state": {"description": "State the city is in", "title": "State", "type": "string"}, "population_millions": {"description": "Population in millions", "title": "Population Millions", "type": "number"}, "famous_for": {"description": "What the city is famous for", "title": "Famous For", "type": "string"}}, "required": ["name", "state", "population_millions", "famous_for"]}
```


In [15]:
# Step 10a: Set up the Pydantic chain
# Goal: Build a chain that validates output with Pydantic
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data extraction assistant. {format_instructions}"),
    ("human", "Extract city info: {text} /no_think")
])
chain = prompt | llm | pydantic_parser

print("Pydantic chain ready!")

Pydantic chain ready!


In [16]:
# Step 10b: Run the Pydantic chain
# Goal: See the validated output with guaranteed types
result = chain.invoke({
    "text": "Hyderabad is the capital of Telangana with about 10 million people. Famous for biryani.",
    "format_instructions": pydantic_parser.get_format_instructions()
})
print(f"City: {result.name}")
print(f"State: {result.state}")
print(f"Population: {result.population_millions}M")
print(f"Famous for: {result.famous_for}")

City: Hyderabad
State: Telangana
Population: 10.0M
Famous for: biryani


<h3 style="text-align: center; color: #097200;">Memory(Converstion History)</h3>

In [17]:
# Step 11a: Manual history -- set up
# Goal: Show how we manage conversation history ourselves
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
history = [
    SystemMessage(content="You are a helpful travel assistant. Keep answers short (1-2 sentences).")
]
# Step 11b: Turn 1 -- ask about Goa
# Goal: First message in the conversation
history.append(HumanMessage(content="I want to plan a trip to Goa next weekend. /no_think"))
response = llm.invoke(history)
print("Turn 1:", response.content)
history.append(response)


Turn 1: Plan a relaxed trip to Goa next weekend, departing around the end of the week. Enjoy beaches like Maligiri, Kerala's temples, and cultural experiences. Check the weather forecast for the best time to plan your itinerary.
